# Power consumption — how it works

This is the explanation and reference for `quicksat`'s power consumption. It covers what the consumption file holds, how the loads follow the scenario's modes, the margins, and where the model stops.

It is not a tutorial. The worked example is `sample/power/power_consumption.ipynb`. This notebook still runs, so that it cannot drift from the code. It reads `docs/power/data/`, its own copy of the inputs. The figures quoted below are true of that data.

The scenario names the modes, and power says what each draws (`docs/explanations/decisions.md`, entry 10). The power budget, `docs/power/power_budget_ref.ipynb`, sets the loads against the generation.

In [1]:
import os
from pathlib import Path

import numpy as np

# Make the in-development quicksat package importable without installing it: walk up
# from the current directory to the repo root (the folder that holds the quicksat
# package) and switch to it. Works whether the notebook runs from docs/, the repo
# root, or the docs build.
here = Path.cwd()
repo_root = next(
    (p for p in (here, *here.parents) if (p / "quicksat" / "__init__.py").exists()),
    None,
)
if repo_root is None:
    raise RuntimeError("could not locate the quicksat repo root")
os.chdir(repo_root)

from quicksat import u
from quicksat.power.consumption import PowerConsumption, run_consumption
from quicksat.scenario.config import SLEW, Scenario
from quicksat.scenario.run import run_scenario

DATA = Path("docs") / "power" / "data"
CONSUMPTION_FILE = DATA / "power_consumption.csv"

consumption = PowerConsumption.from_csv(CONSUMPTION_FILE)
run = run_scenario(Scenario.from_yaml_file(DATA / "scenario.yaml"))
loads = run_consumption(consumption, run)

## The consumption file

The consumption file is a CSV, written by hand, usually `power_consumption.csv`. No notebook writes it. It has one row for each piece of equipment that draws power, so it can be a long file. It is kept apart from the power system file for that reason. Equipment that draws none, such as the structure, is left out. The rows need not match the mass budget's equipment list.

| Column | What it holds |
| --- | --- |
| `equipment_id` | A short name with no spaces, which no other row has |
| `equipment_name` | Free text |
| `subsystem` | Such as `ADCS`, with no spaces. The plots stack the loads by it. |
| `units_on` | How many units draw power: 1 of a cold-redundant pair, or all four reaction wheels |
| `eqpt_margin` | The equipment's margin, in percent, such as `5` |
| one column for each mode | The power of one unit in that mode, such as `10 W`, averaged over the mode |
| `comments` | Optional free text |

A value is an average over the mode. So a heater that a thermostat switches, or a magnetorquer that runs now and then, gives its mean power. Every cell of a mode column needs a power, `0 W` included, so that no mode is ever assumed to draw what another does.

This notebook's file:

In [2]:
print(CONSUMPTION_FILE.read_text())

equipment_id,equipment_name,subsystem,units_on,eqpt_margin,idle,imaging,downlink,slew,comments
obc,SSTL OBC750,OBDH,1,5,10 W,10 W,10 W,10 W,Cold redundant: one of two on
star_tracker,Jena Astro HP optical head,ADCS,2,5,6 W,6 W,6 W,6 W,Both heads on
reaction_wheel,Rockwell RSI 45,ADCS,4,5,10 W,12 W,15 W,10 W,Holding speed; tracking Svalbard changes it most
magnetorquer,ZARM MT30,ADCS,3,5,1 W,1 W,1 W,1 W,Averaged over the momentum dumps
imu,Northrop LN-200S,ADCS,1,5,12 W,12 W,12 W,12 W,
gps_receiver,RUAG PODRIX,ADCS,1,5,10 W,10 W,10 W,10 W,
pcdu,Airbus PCDU housekeeping,EPS,1,10,15 W,15 W,15 W,15 W,Its own electronics; its conversion losses are in power_system.yaml
sadm,Solar array drive,EPS,2,10,4 W,1 W,4 W,4 W,Not in the mass budget's list; holding a parked wing in imaging
s_band_rx,S-band transceiver receivers,COMM,2,5,4 W,4 W,4 W,4 W,Both receivers on for commands
s_band_tx,S-band transceiver transmitter,COMM,1,5,0 W,0 W,15 W,0 W,Telemetry during the station pass
x_band_tx,X-band tra

## The modes, slews included

At each time of the run, each row draws its units' power in the mode of that time. The scenario names the modes in its activities. While the body turns between attitudes, the run gives the mode `slew` instead (see "Slews" in `docs/scenario/scenario_ref.ipynb`). The slew takes the end of the activity before it, and the payload is not at its task while the body turns. So a scenario with slews needs a `slew` column, which only the user can fill.

Every mode the run has needs its column, or the consumption stops with the modes it lacks:

In [3]:
text = CONSUMPTION_FILE.read_text().replace(",slew,", ",turning,")
renamed = DATA.parent / "renamed.csv"
renamed.write_text(text)
try:
    run_consumption(PowerConsumption.from_csv(renamed), run)
except ValueError as error:
    print(error)
finally:
    renamed.unlink()

the consumption file has no column for the modes ['slew'], which the run has. It has columns for ['idle', 'imaging', 'downlink', 'turning'].


A column for a mode the run does not have, such as `safe`, is allowed. The run's `notes` name it, since it is not used. This notebook's run has every mode of the file:

In [4]:
print(sorted({mode for _, _, mode in loads.pieces}))
print(loads.notes or "no notes")

['downlink', 'idle', 'imaging', 'slew']
no notes


## Margins

Each row carries its own margin, as in the mass budget: 5% for equipment that has flown, 10% for equipment that needs changes, and 20% for new designs, such as the focal plane, and for the heaters, whose power depends on the thermal design. The power budget adds a system margin on all the loads together, from the power system file.

All the equipment together, in W:

In [5]:
consumption.mode_totals().style.format("{:.1f}")

,idle [W],imaging [W],downlink [W],slew [W]
without margins,188.0,290.0,293.0,188.0
with equipment margins,207.6,323.9,321.3,207.6


## The loads over a run

The modes change at exact times, between the steps of the grid. So the energy is worked out from the mode intervals, not from the grid: within each interval the power holds steady. `energy_until` gives it at any time, exactly.

`orbit_table` gives each whole orbit, counted from the start of the run as for the generation: the mean and the peak load, the energy drawn, and the part drawn in eclipse. The battery must give that part, through its discharge path. Here the eclipse falls in nadir idle:

In [6]:
loads.orbit_table.style.hide(axis="index").format(
    {
        "mean load [W]": "{:.1f}",
        "peak load [W]": "{:.1f}",
        "energy [Wh]": "{:.1f}",
        "in eclipse [Wh]": "{:.1f}",
    }
)

orbit,start [UTC],mean load [W],peak load [W],energy [Wh],in eclipse [Wh]
1,2026-10-01T09:00:00.000,225.0,323.9,355.7,118.7
2,2026-10-01T10:34:51.055,224.1,323.9,354.3,118.7
3,2026-10-01T12:09:42.109,223.2,323.9,352.9,118.7


The energy over the whole run, from the grid and from the mode intervals. On the grid, each step takes the mode at its start, so it misses where a mode starts or ends between two steps:

In [7]:
seconds = np.asarray((run.times - run.times[0]).to_value(u.s))
exact = loads.energy_until(np.array([seconds[-1]]))[0] / 3600
on_grid = (
    float(np.sum(np.asarray(loads.total.to_value(u.W))[:-1] * np.diff(seconds))) / 3600
)
slewing = sum(end - start for start, end, mode in loads.pieces if mode == SLEW)
print(f"from the mode intervals  {exact:.2f} W h")
print(f"from the grid            {on_grid:.2f} W h")
print(f"time in slews            {slewing / 60:.1f} min")

from the mode intervals  1062.86 W h
from the grid            1063.97 W h
time in slews            27.0 min


## Limitations

What the consumption does not do, yet or at all:

- **Averages, not peaks.** Each mode gives a mean power. Inrush currents, a heater's on and off, or a wheel's spin-up are not followed.
- **One power for each mode.** The power does not depend on the attitude, the illumination or the temperature. Heaters draw the same in sunlight as in eclipse, unless the activities use modes of their own for them.
- **One mode at a time.** Imaging while slewing, such as scanning, is not covered: the run gives the whole slew the mode `slew`.
- **No safe mode by itself.** A safe mode is a mode like any other, with its own column and activities.